In [1]:
# %% [cell 1] Imports
import os
import re
import sys
import math
import json
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression

# && [cell 2] directories
sampled_csv = r"C:\Users\Shahriar Rahman\OneDrive\Desktop\GIS\Dhaka_NO2_Project\Processed_Data\sampled_satellite_no2\sampled_satellite_no2_long.csv"
ground_csv  = r"C:\Users\Shahriar Rahman\OneDrive\Desktop\GIS\Dhaka_NO2_Project\Raw_Data\Ground_CSV\Ground_NO2_monthly.csv"
out_dir = os.path.join(os.path.dirname(sampled_csv), "comparison_outputs")
os.makedirs(out_dir, exist_ok=True)

ROUND_PLACES = 3
RF_N_ESTIMATORS = 100
RF_RANDOM_STATE = 0

print(f"✅ Outputs will be written to: {out_dir}")

# %% [cell 3] Helper Functions
def normalize_ym(val):
    """Convert any date-like value to 'YYYY_MM' format."""
    if pd.isna(val):
        return None
    if isinstance(val, (pd.Timestamp, np.datetime64)):
        dt = pd.to_datetime(val)
        return f"{dt.year}_{dt.month:02d}"
    s = str(val).strip().replace('-', '_')
    m = re.search(r'(\d{4})[_-]?(\d{2})', s)
    if m:
        return f"{m.group(1)}_{m.group(2)}"
    m = re.search(r'(\d{4})(\d{2})', s)
    if m:
        return f"{m.group(1)}_{m.group(2)}"
    try:
        dt = pd.to_datetime(s)
        return f"{dt.year}_{dt.month:02d}"
    except Exception:
        return None

def rmse(a, b):
    a, b = np.array(a, float), np.array(b, float)
    return np.sqrt(np.nanmean((a - b) ** 2))

def mae(a, b):
    a, b = np.array(a, float), np.array(b, float)
    return np.nanmean(np.abs(a - b))

def mape(a, b):
    a, b = np.array(a, float), np.array(b, float)
    denom = np.where(b != 0, np.abs(b), np.nan)
    return np.nanmean(np.abs((a - b) / denom)) * 100

def concordance_ccc(a, b):
    a, b = np.array(a, float), np.array(b, float)
    mask = ~np.isnan(a) & ~np.isnan(b)
    if mask.sum() == 0:
        return np.nan
    a, b = a[mask], b[mask]
    mean_a, mean_b = a.mean(), b.mean()
    var_a, var_b = a.var(ddof=1), b.var(ddof=1)
    sd_a, sd_b = np.sqrt(var_a), np.sqrt(var_b)
    r = np.corrcoef(a, b)[0, 1]
    denom = var_a + var_b + (mean_a - mean_b) ** 2
    return (2 * r * sd_a * sd_b) / denom if denom != 0 else np.nan

# %% [cell 4] Load and Prepare Data
print("📂 Loading data...")

# --- Satellite data ---
sat = pd.read_csv(sampled_csv, dtype=str)
no2_col_sampled = next((c for c in sat.columns if 'no2' in c.lower()), None)
if not no2_col_sampled:
    raise SystemExit("❌ No NO2 column found in satellite CSV.")

ym_col_sampled = next((c for c in sat.columns if c.lower() in ('year_month','ym','date','datetime','month','time')), None)
if not ym_col_sampled:
    raise SystemExit("❌ No date/month column found in satellite CSV.")

sat['ym'] = sat[ym_col_sampled].apply(normalize_ym)
sat['NO2_sat'] = pd.to_numeric(sat[no2_col_sampled], errors='coerce')
sat_monthly = (
    sat.dropna(subset=['ym','NO2_sat'])
    .groupby('ym')['NO2_sat']
    .mean()
    .reset_index()
    .rename(columns={'NO2_sat':'NO2_sat_mean'})
)

# --- Ground data ---
g = pd.read_csv(ground_csv, dtype=str)
date_col_ground = next((c for c in g.columns if c.lower() in ('year_month','ym','date','datetime','month','time')), None)
if not date_col_ground:
    raise SystemExit("❌ No date/month column found in ground CSV.")
no2_col_ground = next((c for c in g.columns if 'no2' in c.lower()), None)
if not no2_col_ground:
    raise SystemExit("❌ No NO2 column found in ground CSV.")

g['ym'] = g[date_col_ground].apply(normalize_ym)
g[no2_col_ground] = pd.to_numeric(g[no2_col_ground], errors='coerce')
ground_monthly = g[['ym', no2_col_ground]].dropna().rename(columns={no2_col_ground:'NO2_ground_mean'})

# --- Merge ---
df_monthly = pd.merge(ground_monthly, sat_monthly, on='ym', how='inner').dropna()
if df_monthly.empty:
    raise SystemExit("❌ No overlapping months found. Check your date formats.")

df_monthly = df_monthly.round(ROUND_PLACES)
pairs_csv = os.path.join(out_dir, "comparison_results.csv")
df_monthly.to_csv(pairs_csv, index=False)
print(f"✅ Saved monthly pairs: {pairs_csv}\n", df_monthly.head())

# %% [cell 5] Metrics Calculation
sat_vals = df_monthly['NO2_sat_mean'].values
ground_vals = df_monthly['NO2_ground_mean'].values

metrics = {
    'n_months': len(df_monthly),
    'rmse': round(rmse(sat_vals, ground_vals), ROUND_PLACES),
    'mae': round(mae(sat_vals, ground_vals), ROUND_PLACES),
    'mape_percent': round(mape(sat_vals, ground_vals), ROUND_PLACES),
    'bias_mean': round(np.nanmean(sat_vals - ground_vals), ROUND_PLACES),
    'pearson_r': round(np.corrcoef(sat_vals, ground_vals)[0,1], ROUND_PLACES),
    'r_squared': round(np.corrcoef(sat_vals, ground_vals)[0,1]**2, ROUND_PLACES),
    'ccc': round(concordance_ccc(sat_vals, ground_vals), ROUND_PLACES)
}
summary_csv = os.path.join(out_dir, "statistical_closeness_summary.csv")
pd.DataFrame([metrics]).to_csv(summary_csv, index=False)
print("📊 Overall metrics:\n", json.dumps(metrics, indent=2))
print(f"✅ Saved summary CSV: {summary_csv}")

# %% [cell 6] Visualizations
# Scatter plot
plt.figure(figsize=(6,6))
plt.scatter(ground_vals, sat_vals, alpha=0.7, edgecolor='k', linewidth=0.2)
lims = [0, max(np.nanmax(ground_vals), np.nanmax(sat_vals)) * 1.05]
plt.plot(lims, lims, 'k--', label='1:1 Line')
plt.xlabel('Ground NO₂ (µg/m³)')
plt.ylabel('Satellite NO₂ (µg/m³)')
plt.title(f"Satellite vs Ground (Monthly Means, N={len(df_monthly)})")
plt.legend(); plt.grid(True, linestyle=':', linewidth=0.5)
scatter_png = os.path.join(out_dir, "scatter_sat_vs_ground.png")
plt.tight_layout(); plt.savefig(scatter_png, dpi=200); plt.close()
print(f"✅ Saved scatter plot: {scatter_png}")

# Time series plot
df_sorted = df_monthly.copy()
df_sorted['ym_dt'] = pd.to_datetime(df_sorted['ym'].str.replace('_','-') + '-01')
df_sorted = df_sorted.sort_values('ym_dt')
plt.figure(figsize=(10,4))
plt.plot(df_sorted['ym_dt'], df_sorted['NO2_sat_mean'], marker='o', label='Satellite Mean')
plt.plot(df_sorted['ym_dt'], df_sorted['NO2_ground_mean'], marker='o', label='Ground Mean')
plt.xlabel('Month'); plt.ylabel('NO₂ (µg/m³)')
plt.title('Monthly NO₂ Comparison: Satellite vs Ground')
plt.legend(); plt.grid(True, linestyle=':', linewidth=0.5)
plt.xticks(rotation=45)
ts_png = os.path.join(out_dir, "monthly_timeseries.png")
plt.tight_layout(); plt.savefig(ts_png, dpi=200); plt.close()
print(f"✅ Saved time-series plot: {ts_png}")

# %% [cell 7] Leave-One-Month-Out (LOMO) Cross-Validation
months = df_monthly['ym'].tolist()
if len(months) < 3:
    print("⚠️ Not enough months for CV (need ≥3). Skipping CV.")
else:
    X = df_monthly['NO2_sat_mean'].values.reshape(-1, 1)
    y = df_monthly['NO2_ground_mean'].values
    cv_records, rf_records = [], []

    for i, left_out in enumerate(months):
        train_idx = [j for j in range(len(months)) if j != i]
        test_idx = [i]

        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        # Linear regression
        lin = LinearRegression().fit(X_train, y_train)
        lr_pred = lin.predict(X_test)[0]

        # Random Forest
        rf = RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, random_state=RF_RANDOM_STATE)
        rf.fit(X_train, y_train)
        rf_pred = rf.predict(X_test)[0]

        cv_records.append({
            'left_out_ym': left_out,
            'model': 'LinearRegression',
            'true': round(float(y_test[0]), ROUND_PLACES),
            'predicted': round(float(lr_pred), ROUND_PLACES),
            'error': round(float(lr_pred - y_test[0]), ROUND_PLACES)
        })
        rf_records.append({
            'left_out_ym': left_out,
            'model': 'RandomForest',
            'true': round(float(y_test[0]), ROUND_PLACES),
            'predicted': round(float(rf_pred), ROUND_PLACES),
            'error': round(float(rf_pred - y_test[0]), ROUND_PLACES)
        })

    df_cv = pd.concat([pd.DataFrame(cv_records), pd.DataFrame(rf_records)], ignore_index=True)

    # CV summary
    cv_summary = []
    for model, group in df_cv.groupby('model'):
        true, pred = group['true'], group['predicted']
        cv_summary.append({
            'model': model,
            'n_folds': len(group),
            'rmse_cv': round(rmse(pred, true), ROUND_PLACES),
            'mae_cv': round(mae(pred, true), ROUND_PLACES),
            'r2_cv': round(r2_score(true, pred), ROUND_PLACES),
            'pearson_r_cv': round(np.corrcoef(pred, true)[0,1], ROUND_PLACES),
            'mape_percent_cv': round(mape(pred, true), ROUND_PLACES),
            'ccc_cv': round(concordance_ccc(pred, true), ROUND_PLACES)
        })

    excel_path = os.path.join(out_dir, "cv_results.xlsx")
    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        df_monthly.to_excel(writer, sheet_name='monthly_stats', index=False)
        df_cv.to_excel(writer, sheet_name='cv_predictions', index=False)
        pd.DataFrame(cv_summary).to_excel(writer, sheet_name='cv_summary', index=False)

    print("✅ Saved CV results to:", excel_path)
    print("📘 CV Summary:\n", pd.DataFrame(cv_summary))

# %% [cell 8] Done
print("\n🎯 All done! Outputs saved in:", out_dir)
for f in os.listdir(out_dir):
    print(" -", f)


✅ Outputs will be written to: C:\Users\Shahriar Rahman\OneDrive\Desktop\GIS\Dhaka_NO2_Project\Processed_Data\sampled_satellite_no2\comparison_outputs
📂 Loading data...
✅ Saved monthly pairs: C:\Users\Shahriar Rahman\OneDrive\Desktop\GIS\Dhaka_NO2_Project\Processed_Data\sampled_satellite_no2\comparison_outputs\comparison_results.csv
         ym  NO2_ground_mean  NO2_sat_mean
0  2019_01           39.581        49.302
1  2019_02           38.799        45.464
2  2019_03           38.942        34.263
3  2019_04           39.308        19.475
4  2019_05           39.323        15.982
📊 Overall metrics:
 {
  "n_months": 72,
  "rmse": 18.222,
  "mae": 15.513,
  "mape_percent": 37.719,
  "bias_mean": -7.134,
  "pearson_r": -0.003,
  "r_squared": 0.0,
  "ccc": -0.0
}
✅ Saved summary CSV: C:\Users\Shahriar Rahman\OneDrive\Desktop\GIS\Dhaka_NO2_Project\Processed_Data\sampled_satellite_no2\comparison_outputs\statistical_closeness_summary.csv
✅ Saved scatter plot: C:\Users\Shahriar Rahman\OneDrive